## Objective : 
Analyze retail sales data to identify top-performing products, regional trends, and key business KPIs (Revenue, Profit, Growth rate, customer segmentation)

## Dataset : 
Sample Superstore dataset - [9994] rows, [21] columns

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('../data/Sample _ Superstore.csv')

## 1. Data Exploration
Checking structure ,data types and summary statistics before cleaning

In [ ]:
df.head()


In [ ]:
df.shape


In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# per-column missing counts
df.isnull().sum()


**Finding:** 
No missing values found across all columns

In [ ]:
# Data cleaning 
# checks duplicate rows
df.duplicated().sum()

**Finding:**
No duplicate rows found,no action needed

## 2. Data Cleaning:
Converting Order Date and Ship Date from string to datetime for time-based analysis.

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])   

## 3. Overall Business KPIs:

In [ ]:
total_sales = round(df["Sales"].sum(), 2)
total_profit = round(df["Profit"].sum(), 2)
total_orders = df["Order ID"].nunique()
avg_order_value = round(total_sales / total_orders, 2)
profit_margin = round((total_profit / total_sales) * 100, 2)

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Orders: {total_orders}")
print(f"Average Order Value: ${avg_order_value:,.2f}")
print(f"Overall Profit Margin: {profit_margin}%")


In [ ]:
avg_sales = round(df["Sales"].mean(), 2)
avg_profit = round(df["Profit"].mean(), 2)

print(f"Average Sales per transaction: ${avg_sales:,.2f}")
print(f"Average Profit per transaction: ${avg_profit:,.2f}")

**Summary:**
Total Revenue: $2,297,200.86 | Total Profit: $286,397.02 | Total Orders: 5,009 | Average Order Value: $458.61 | Overall Profit Margin: 12.47%


##  Visualizations


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 4. Category-Level Analysis
Comparing sales, profit, and discount behaviour across product categories.


In [ ]:
df.groupby("Category")["Sales"].sum().round(2).sort_values(ascending=False)


In [ ]:
df.groupby("Category")["Profit"].sum().round(2).sort_values(ascending=False)

In [ ]:
category_margin = (df.groupby("Category")["Profit"].sum() / df.groupby("Category")["Sales"].sum() * 100).round(2)
category_margin.sort_values(ascending=False)

**Insight:** Technology (17.40%) and Office Supplies (17.04%) have healthy, comparable
profit margins, but Furniture lags at just 2.49% — nearly 7x lower despite
contributing meaningfully to overall sales. This suggests Furniture's revenue isn't
converting into profit efficiently.

In [ ]:
cat_margin = (df.groupby("Category")["Profit"].sum() / df.groupby("Category")["Sales"].sum() * 100).round(2).sort_values()

colors = ["crimson" if v < 5 else "seagreen" for v in cat_margin]
cat_margin.plot(kind="barh", color=colors)
plt.title("Profit Margin by Category (%)")
plt.xlabel("Profit Margin (%)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

**Chart Insight:** You can literally see it here - Furniture's bar barely gets off the ground compared to Technology and Office Supplies. Makes the 7x margin gap a lot easier to process than the raw percentages.

In [ ]:
sub_category_margin = (df.groupby("Sub-Category")["Profit"].sum() / df.groupby("Sub-Category")["Sales"].sum() * 100).round(2)
sub_category_margin.sort_values(ascending=False)

**Insight:** Three sub-categories are actively losing money: Tables (-8.56%),
Bookcases (-3.02%), and Supplies (-2.55%). Tables and Bookcases explain Furniture's
poor category-level margin, but Supplies is a hidden problem — Office Supplies looks
healthy overall (17.04%) only because strong performers like Labels (44.42%) and
Paper (43.39%) mask the loss happening in Supplies. Category-level averages alone
would have missed this; sub-category analysis was necessary to find it

In [ ]:
sub_margin = (df.groupby("Sub-Category")["Profit"].sum() / df.groupby("Sub-Category")["Sales"].sum() * 100).round(2).sort_values()

colors = ["crimson" if v < 0 else "steelblue" for v in sub_margin]
sub_margin.plot(kind="barh", color=colors, figsize=(8,8))
plt.title("Profit Margin by Sub-Category (%)")
plt.xlabel("Profit Margin (%)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

**Chart Insight:** Only three bars dip below zero - Tables, Bookcases and Supplies - and Tables is clearly the worst offender by a good margin. Everything above Chairs is at least a healthy double-digit margin, so the problem really is isolated to a handful of sub-categories, not the whole business.

**Recommendation:** Review discount policy specifically for Tables, Bookcases, and
Supplies — either reduce discount depth on these items or reassess their pricing,
since they are eroding profit gained elsewhere in the business.

In [ ]:
(df.groupby("Category")["Discount"].mean()*100).round(2).sort_values(ascending=False)  

**Insight:** Furniture carries the highest average discount (17.39%) among all categories
and the lowest profit margin (2.49%) — nearly 7x lower than Technology and Office Supplies,
which both maintain healthy ~17% margins despite Technology having the lowest discount
(13.23%). 

In [ ]:
sns.scatterplot(data=df, x="Discount", y="Profit", hue="Category", alpha=0.6)
plt.title("Discount vs Profit by Category")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

**Chart Insight:** Once discount goes past roughly 20-30%, profit points start dropping below the zero line for basically every category, and by 70-80% discount almost everything is a loss. Furniture (blue) and Technology (green) have some of the biggest losses at high discount levels, which lines up with the margin numbers above - discounting that deep is basically handing away all the profit.

## 5. Regional Analysis
Comparing sales and profit performance across regions and states.

In [ ]:
df.groupby("Region")["Sales"].sum().round(2).sort_values(ascending=False)


In [ ]:
df.groupby("Region")["Profit"].sum().round(2).sort_values(ascending=False)



In [ ]:
df.groupby("State")["Sales"].sum().round(2).sort_values(ascending=False)


In [ ]:
df.groupby("State")["Profit"].sum().round(2).sort_values(ascending=False)

**Insight:** California is the single highest-revenue state, and the West region leads on both sales and profit overall. However, sales and profit don't always move together at the state level — several states generate solid sales but comparatively thin profit, pointing to pricing or cost-efficiency issues specific to those markets rather than a demand problem.


In [ ]:
region_summary = df.groupby("Region")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False)
region_summary.plot(kind="bar")
plt.title("Sales vs Profit by Region")
plt.ylabel("Amount ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Chart Insight:** West and East are ahead on both sales and profit, no surprise there. But look closely at Central vs South - Central actually has higher sales than South, yet South pulls in more profit. So Central isn't just behind, it's behind on efficiency too, not just size.

**Follow-up:** Chairs is a mid-pack sub-category by margin (positive but modest). Pulling one specific high-ticket chair product checks whether the discount-erodes-profit pattern found above holds at the individual order level, not just in aggregate.


In [ ]:
df[df["Product Name"] == "HON 5400 Series Task Chairs for Big and Tall"][["Sales", "Discount", "Profit", "Quantity"]]

**Insight:** The pattern holds at the individual order level. The same chair is profitable at 0% and 10% discount (+$700.98 and +$420.59), breaks even at 20% (two orders at exactly $0 profit), and loses money at 30% (-$140 to -$631 across three orders). A single product can swing from a solidly profitable item to a loss-maker purely based on the discount applied to it.


## 6. Top Customers and Products
Identifying the highest-value customers and best-selling products by sales.

In [ ]:
# Customer analysis
df.groupby("Customer Name")["Sales"].sum().round(2).sort_values(ascending=False).head(10)

In [ ]:
#Product analysis
df.groupby("Product Name")["Sales"].sum().round(2).sort_values(ascending=False).head(10)

In [ ]:
top_10_products = df.groupby("Product Name")["Sales"].sum().sort_values(ascending=False).head(10).index

df[df["Product Name"].isin(top_10_products)].groupby("Product Name").agg(
    Category=("Category", "first"),
    Sub_Category=("Sub-Category", "first"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).round(2).sort_values("Total_Sales", ascending=False)

In [ ]:
top_products = (
    df.groupby("Product Name")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

plt.figure(figsize=(10,6))
plt.barh(top_products.index, top_products.values)
plt.title("Top 10 Products by Sales")
plt.xlabel("Sales")
plt.tight_layout()
plt.show()

**Chart Insight:** Canon imageCLASS copier is way out in front of everything else, almost double the next closest product. After the top 2-3 products it flattens out quite a bit, so the top sellers list isn't as top-heavy as it might look from the table alone.

**Follow-up:** Since discount depth showed up as a major profit driver above, it's worth checking whether that relationship holds within a single sub-category — grouping Chairs orders by discount band isolates the effect from category-level noise.


In [ ]:
df[df["Sub-Category"] == "Chairs"].groupby("Discount")["Profit"].agg(["mean", "count"]).round(2)

**Insight:** Within Chairs alone, mean profit per order falls in a straight line as discount increases — $164.91 at 0% discount, $93.57 at 10%, $17.13 at 20%, and a loss of -$42.64 at 30% (based on 158 orders at that level, so this isn't a fluke). This is the clearest single piece of evidence that discounting past a certain point isn't just correlated with lower margins — it's actively causing the sub-category-level losses seen earlier.


**Insights:**
A small number of customers contribute a significant share of total revenue.


## 7. Customer Segment Analysis
Comparing sales and profit across customer segments (Consumer, Corporate, Home Office).


In [ ]:
df.groupby("Segment")["Sales"].sum().round(2).sort_values(ascending=False)



In [ ]:
df.groupby("Segment")["Profit"].sum().round(2).sort_values(ascending=False)

In [ ]:
segment_sales = df.groupby("Segment")["Sales"].sum()

plt.figure(figsize=(6,4))
plt.bar(segment_sales.index, segment_sales.values)
plt.title("Sales by Customer Segment")
plt.ylabel("Sales")
plt.show()

**Chart Insight:** Consumer segment towers over the other two - it's not even close. So most of the revenue is coming from individual buyers rather than businesses, which is worth keeping in mind for any retention or marketing push.

**Insights:**
Consumers lead both sales and profit

## 8. Shipping Method Analysis
Comparing sales volume across shipping modes.

In [ ]:

df.groupby("Ship Mode")["Sales"].sum().round(2).sort_values(ascending=False) 

In [ ]:
ship_sales = df.groupby("Ship Mode")["Sales"].sum()

plt.figure(figsize=(7,4))
plt.bar(ship_sales.index, ship_sales.values)
plt.title("Sales by Shipping Mode")
plt.ylabel("Sales")
plt.xticks(rotation=20)
plt.show()

**Chart Insight:** Standard Class completely dwarfs the other shipping options. Most customers clearly aren't in a rush, which is useful to know if fulfillment costs ever need optimizing.

**Insights:**
 Standard Class dominates ($1.36M vs. next-highest $459K for Second Class) — most orders aren't urgent.


In [ ]:
df["Shipping Days"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

shipping_delay = (
    df.groupby("Ship Mode")["Shipping Days"]
      .mean()
      .sort_values()
)

shipping_delay

## 9. Time-Based Trend Analysis

In [ ]:
df["Month"] = df["Order Date"].dt.month_name()
df["Year"] = df["Order Date"].dt.year

In [ ]:
df["Month Number"] = df["Order Date"].dt.month
monthly_sales = df.groupby(["Year", "Month Number", "Month"])["Sales"].sum().round(2).reset_index()


In [ ]:
monthly_sales = monthly_sales.sort_values(["Year", "Month Number"]).reset_index(drop=True)
monthly_sales["Growth Rate (%)"] = (
    monthly_sales["Sales"]
        .pct_change()
        .mul(100)
        .round(2)
)

monthly_sales

In [ ]:
monthly_sales["Period"] = monthly_sales["Year"].astype(str) + "-" + monthly_sales["Month Number"].astype(str).str.zfill(2)

plt.plot(monthly_sales["Period"], monthly_sales["Sales"], marker="o", markersize=3)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

**Insight:** Sales clearly trend upward from 2014 to 2017 overall, but it's not a smooth climb - there's a repeating pattern of a big spike around Nov-Dec each year followed by a January-February dip. Looks like a seasonal, holiday-driven pattern rather than steady growth, which matters for inventory planning.

## 10. Negative Profit Products

In [ ]:
negative_products = (
    df.groupby("Product Name")["Profit"]
      .sum()
      .reset_index()
)
negative_products = negative_products[
    negative_products["Profit"] < 0
].sort_values("Profit")

negative_products.head(15)

# Final Business Recommendations

1. Reduce excessive discounts on Furniture, especially Tables and Bookcases, to improve profitability.

2. Investigate loss-making sub-categories — Tables (-8.56%), Bookcases (-3.02%), and Supplies (-2.55%) — and optimize their pricing strategy; Supplies in particular is masked by a healthy Office Supplies category average.

3. Continue investing in high-performing Technology products to maximize revenue and profit.

4. Strengthen customer retention initiatives for high-value customers.

5. Increase marketing efforts in high-performing regions while developing targeted strategies for underperforming markets.

6. Use monthly sales trends to improve inventory planning and seasonal promotions.

In [ ]:
df.to_csv("../data/cleaned_superstore.csv", index=False)